In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class MHA(nn.Module):
    def __init__(self, hidden_dim, num_head, dropout_rate=0.1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_head = num_head
        assert self.hidden_dim%self.num_head == 0
        self.head_dim = self.hidden_dim//self.num_head
        self.dropout_rate = dropout_rate

        self.q_proj = nn.Linear(self.hidden_dim, self.hidden_dim)
        self.k_proj = nn.Linear(self.hidden_dim, self.hidden_dim)
        self.v_proj = nn.Linear(self.hidden_dim, self.hidden_dim)
        self.dropout = nn.Dropout(self.dropout_rate)
        self.o_proj = nn.Linear(self.hidden_dim, self.hidden_dim)

    def forward(self, x, mask=None):
        # x shape  [b,s,hidden_dim]
        b,s = x.shape[0], x.shape[1]
        
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(b,s,self.num_head,self.head_dim).transpose(1,2)
        k = k.view(b,s,self.num_head,self.head_dim).transpose(1,2)
        v = v.view(b,s,self.num_head,self.head_dim).transpose(1,2)

        atten = (q @ k.transpose(-1,-2)) /(math.sqrt(self.head_dim))
        if mask is not None:
            atten = atten.masked_fill(mask==0,float("-inf"))
        
        atten = self.dropout(F.softmax(atten,dim=-1))
        # shape  [b, num_head, s, hidden_dim]
        output = atten @ v
        output = output.transpose(1,2).contiguous().view(b,s,-1)
        output = self.o_proj(output)

        return output

x = torch.rand(4,8,64)
net = MHA(64,8)

out = net(x)
print(x.shape)
print(out.shape)

mask1 = torch.ones(4,8)
mask1[:,4:] = 0
pad_mask = mask1.unsqueeze(1).unsqueeze(1)
mask2 = torch.tril(torch.ones(8,8))
causal_mask = mask2.unsqueeze(0).unsqueeze(0)
mask = pad_mask * causal_mask

print(net(x, mask).shape)

torch.Size([4, 8, 64])
torch.Size([4, 8, 64])
torch.Size([4, 8, 64])


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class MQA(nn.Module):
    def __init__(self, hidden_dim, num_head, dropout_rate=0.1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_head = num_head
        assert self.hidden_dim % self.num_head == 0
        self.head_dim = self.hidden_dim // self.num_head
        self.dropout_rate = dropout_rate

        self.q_proj = nn.Linear(self.hidden_dim, self.hidden_dim)
        self.k_proj = nn.Linear(self.hidden_dim, self.head_dim)
        self.v_proj = nn.Linear(self.hidden_dim, self.head_dim)
        self.dropout = nn.Dropout(self.dropout_rate)
        self.o_proj = nn.Linear(self.hidden_dim, self.hidden_dim)

    def forward(self, x, mask=None):
        # x shape  [b,s,hidden_dim]
        b,s = x.shape[0], x.shape[1]

        # [b,s,hidden_dim]
        q = self.q_proj(x)
        # [b,s,head_dim]
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(b,s,self.num_head,self.head_dim).transpose(1,2)

        k = k.unsqueeze(1).expand(-1, self.num_head, -1, -1)
        v = v.unsqueeze(1).expand(-1, self.num_head, -1, -1)

        atten = (q @ k.transpose(-1,-2)) /(math.sqrt(self.head_dim))
        if mask is not None:
            atten = atten.masked_fill(mask==0,float("-inf"))
        
        atten = self.dropout(F.softmax(atten,dim=-1))
        # shape  [b, num_head, s, hidden_dim]
        output = atten @ v
        output = output.transpose(1,2).contiguous().view(b,s,-1)
        output = self.o_proj(output)

        return output

x = torch.rand(4,8,64)
net = MQA(64,8)

out = net(x)
print(x.shape)
print(out.shape)

mask1 = torch.ones(4,8)
mask1[:,4:] = 0
pad_mask = mask1.unsqueeze(1).unsqueeze(1)
mask2 = torch.tril(torch.ones(8,8))
causal_mask = mask2.unsqueeze(0).unsqueeze(0)
mask = pad_mask * causal_mask

print(net(x, mask).shape)

torch.Size([4, 8, 64])
torch.Size([4, 8, 64])
torch.Size([4, 8, 64])


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


class GQA(nn.Module):
    def __init__(self, hidden_dim, num_head, num_group, dropout=0.1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_head = num_head
        self.num_group = num_group

        assert hidden_dim % num_head == 0
        assert num_head % num_group == 0
        self.dropout = dropout
        self.head_dim = hidden_dim // num_head

        self.q_proj = nn.Linear(hidden_dim,hidden_dim)
        self.k_proj = nn.Linear(hidden_dim,self.head_dim * self.num_group)
        self.v_proj = nn.Linear(hidden_dim,self.head_dim * self.num_group)

        self.Dropout = nn.Dropout(self.dropout)
        self.o_proj = nn.Linear(hidden_dim,hidden_dim)

    def forward(self, x, mask=None):
        # x [b,s,hidden_dim]
        b,s = x.shape[0], x.shape[1]

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(b,s,self.num_head,self.head_dim).transpose(1,2)
        k = k.view(b,s,self.num_group,self.head_dim).transpose(1,2)
        v = v.view(b,s,self.num_group,self.head_dim).transpose(1,2)
        k = k.repeat_interleave(self.num_head//self.num_group, dim=1)
        v = v.repeat_interleave(self.num_head//self.num_group, dim=1)

        atten = torch.matmul(q,k.transpose(-1,-2))/math.sqrt(self.head_dim)
        if mask is not None:
            atten = atten.masked_fill(mask==0,-1e9)
        
        atten = self.Dropout(F.softmax(atten,-1))

        out = atten @ v
        # [b,num_head,s, head_dim]
        out = out.transpose(1,2).contiguous().view(b,s,-1)
        out = self.o_proj(out)

        return out

b, s, hidden_dim = 4, 8, 128
net = GQA(128,8,4)
x = torch.rand(b,s,hidden_dim)

pad_mask = torch.ones(b,s)
pad_mask[:,6:] = 0
pad_mask = pad_mask.unsqueeze(1).unsqueeze(1)

causal_mask = torch.tril(torch.ones(s,s))
causal_mask = causal_mask.unsqueeze(0).unsqueeze(0)

# [b,1,1,s] * [1,1,s,s]
mask = pad_mask * causal_mask
print(net(x,mask).shape)


torch.Size([4, 8, 128])


In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class MHA_kv_Cache(nn.Module):
    def __init__(self, hidden_dim, num_head, dropout_rate=0.1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_head = num_head
        assert self.hidden_dim%self.num_head == 0
        self.head_dim = self.hidden_dim//self.num_head
        self.dropout_rate = dropout_rate
        self.k_cache = None
        self.v_cache = None

        self.q_proj = nn.Linear(self.hidden_dim, self.hidden_dim)
        self.k_proj = nn.Linear(self.hidden_dim, self.hidden_dim)
        self.v_proj = nn.Linear(self.hidden_dim, self.hidden_dim)
        self.dropout = nn.Dropout(self.dropout_rate)
        self.o_proj = nn.Linear(self.hidden_dim, self.hidden_dim)

    def forward(self, x, mask=None, use_cache=False):
        # x shape  [b,s,hidden_dim]
        b,s = x.shape[0], x.shape[1]
        
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(b,s,self.num_head,self.head_dim).transpose(1,2)
        k = k.view(b,s,self.num_head,self.head_dim).transpose(1,2)
        v = v.view(b,s,self.num_head,self.head_dim).transpose(1,2)

        if use_cache and self.k_cache is not None:
            k = torch.cat((self.k_cache,k),dim=-2)
            v = torch.cat((self.v_cache,v),dim=-2)

        if use_cache:
            self.k_cache = k
            self.v_cache = v

        atten = (q @ k.transpose(-1,-2)) /(math.sqrt(self.head_dim))
        if mask is not None:
            atten = atten.masked_fill(mask==0,float("-inf"))
        
        atten = self.dropout(F.softmax(atten,dim=-1))
        # shape  [b, num_head, s, hidden_dim]
        output = atten @ v
        output = output.transpose(1,2).contiguous().view(b,s,-1)
        output = self.o_proj(output)

        return output

# 🔧 测试
x1 = torch.rand(1, 1, 128)
x2 = torch.rand(1, 1, 128)
net = MHA_kv_Cache(128, 8)

# 第一次前向传播（建立缓存）
out1 = net(x1, use_cache=True)
print("out1:", out1.shape, "| cache_k:", net.k_cache.shape)

# 第二次前向传播（复用缓存）
out2 = net(x2, use_cache=True)
print("out2:", out2.shape, "| cache_k:", net.k_cache.shape)

# 第二次前向传播（复用缓存）
out2 = net(x2, use_cache=True)
print("out2:", out2.shape, "| cache_k:", net.k_cache.shape)

out1: torch.Size([1, 1, 128]) | cache_k: torch.Size([1, 8, 1, 16])
out2: torch.Size([1, 1, 128]) | cache_k: torch.Size([1, 8, 2, 16])
out2: torch.Size([1, 1, 128]) | cache_k: torch.Size([1, 8, 3, 16])
